<a href="https://colab.research.google.com/github/TWalkerSMCM/icsa-scraper/blob/main/examples/sailor-head-to-head.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# icsa-scraper · Sailor Head-to-Head

Reproduce the **Sailor Head-to-Head** page from the [college sailing analytics site](https://collegeanalytics.goforthom.as) — pick two sailors and see every regatta and race they've sailed against each other, across seasons.

The webapp answers this by loading a SQLite dump of the *entire* database and joining on sailor IDs. Here we do it live against [scores.collegesailing.org](https://scores.collegesailing.org) with `scraper.head_to_head()` — no database, no season scrape.


## Install

In [ ]:
# Fresh runtimes (Colab) need the library. Guard on the distribution name so
# re-running locally can't clobber an editable dev install; to force-update a
# cached Colab runtime, restart it (or run the %pip line with --force-reinstall).
from importlib.metadata import PackageNotFoundError, version

try:
    version("icsa-scraper")
except PackageNotFoundError:
    %pip install -q "icsa-scraper[fetch] @ git+https://github.com/TWalkerSMCM/icsa-scraper"

## The trick: one profile page is a whole career

Every sailor has a profile page (`urls.sailor_profile(slug)` → `/sailors/{slug}/`) listing **every regatta they've ever sailed**, with role, division, and finishing place — spanning every season, not just one.

That means comparing two sailors doesn't require scraping a season (or several) and cross-referencing rosters. It costs exactly **2 requests**: fetch A's profile, fetch B's profile, intersect the regatta-divisions they share. `scraper.head_to_head(a, b)` does exactly that. Only if you want race-by-race detail does it load the (small) set of shared regattas — still far cheaper than a season sweep.


## Find two sailors to compare

`head_to_head` takes sailor **slugs**, not names, so we need two real ones. Rather than hardcoding a pair, we scrape one big regatta — a national championship, lots of entries — and pick the two skippers who sailed the *most races in the same division*. That guarantees they actually raced each other, and it's the same discovery trick the quickstart notebook uses.


In [ ]:
from collections import defaultdict

import scraper

SEASON = "s25"
REGATTA = "open-fleet-race-national-championships"

data = scraper.load(SEASON, only=[REGATTA])

# Group skippers by (regatta, division); take the division with the most
# distinct skippers, then any two of them are guaranteed to have raced together.
by_div = defaultdict(set)
for r in data.sailor_races:
    if r.boat_role == "skipper":
        by_div[(r.regatta_slug, r.division)].add(r.sailor_slug)

(regatta_slug, division), skippers = max(by_div.items(), key=lambda kv: len(kv[1]))
a, b = sorted(skippers)[:2]

print(f"{len(skippers)} skippers in {regatta_slug} division {division}")
print("Comparing:", a, "vs", b)

## Regatta-level comparison — 2 requests, no season load

`head_to_head(a, b)` fetches both profiles and intersects every regatta-division they've *ever* both sailed — not just the one we found them in above. `.shared_frame()` is the pandas escape hatch out of `.shared`; `a_ahead` / `b_ahead` tally regatta-divisions where one sailor placed ahead of the other.


In [ ]:
h = scraper.head_to_head(a, b)

print(f"{a} vs {b} — {len(h.shared)} shared regatta-divisions across their careers")
print(f"{a} ahead in {h.a_ahead}, {b} ahead in {h.b_ahead}")

h.shared_frame().sort_values("season")

## Race-level comparison — loads only the shared regattas

Passing `races=True` goes one level deeper: it loads *just* the shared regattas (via `load_regattas`, not a full season scrape) and finds every individual race both sailors were on the water for. `.races_frame()` is the pandas view; `a_race_wins` / `b_race_wins` count races where one beat the other outright (lower place wins).


In [ ]:
h2 = scraper.head_to_head(a, b, races=True)

print(f"{len(h2.races)} shared races")
print(f"{a} won {h2.a_race_wins}, {b} won {h2.b_race_wins}")

races = h2.races_frame()
races.head(10)

## Chart: every shared race, A's place vs. B's place

One dot per shared race. Points **below** the diagonal are races A won (lower place); points **above** are races B won. Points on the line are ties (rare — usually a shared DNF/DSQ score).


In [ ]:
import matplotlib.pyplot as plt

lo = min(races["place_a"].min(), races["place_b"].min())
hi = max(races["place_a"].max(), races["place_b"].max())

plt.figure(figsize=(6, 6))
plt.plot([lo, hi], [lo, hi], linestyle="--", color="gray", linewidth=1, label="tie line")
plt.scatter(races["place_a"], races["place_b"], alpha=0.6)
plt.xlabel(f"{a} — place")
plt.ylabel(f"{b} — place")
plt.title(f"{a} vs {b} — every shared race\n(below the line = {a} won, above = {b} won)")
plt.gca().invert_xaxis()  # lower place (better finish) reads left-to-right
plt.gca().invert_yaxis()
plt.legend()
plt.tight_layout()
plt.show()

---
**Recap:** `scraper.head_to_head(a, b)` compares two sailors' full careers for the cost of two profile fetches — `.shared` / `.shared_frame()` for the regatta-division overlap, `.a_ahead` / `.b_ahead` for the tally. Add `races=True` for race-by-race detail (`.races` / `.races_frame()`, `.a_race_wins` / `.b_race_wins`), which loads only the shared regattas, not a whole season.

**Other angles to try:**
- Scope the comparison to one season or division of interest by filtering `h.shared_frame()` / `h.races_frame()` on `season`/`division` after the fact.
- For a season-wide view instead of one pair of sailors — e.g. every skipper's record against every other skipper in a division — load the season directly with `scraper.load(season, only=[...])` and cross-reference `data.sailor_races` yourself; `head_to_head` is optimized for the *two-sailor* case, not an all-pairs sweep.
- `h2.races_frame()` is already a chronological (well, sortable by season) sequence of head-to-head finishes — plug it into a simple ELO/Elo-like update step to track how one rivalry's ratings moved over time.
